# 03 NEMO: the Eocene ocean

Two steps of the Eocene pipeline change the ocean:

* `domain` (`EoceneDomain`) installs the Eocene PALEORCA2 domain (`domain_cfg.nc`, `maskutil.nc`) and checks that
  it has the same vertical grid as the present-day domain;
* `nemo` (`EoceneNEMO`) rewrites the forcing and initial files: tidal mixing, geothermal heat flux, ocean initial
  state and river runoff. It reads only the present-day domains, on purpose: hence the vertical grid check.

The Eocene domain itself is built by the NEMO domain pipeline (`ecefast nemo domain`): coordinates, bathymetry,
namelists, then DOMAINcfg, which runs only on ECMWF Atos. On a laptop the `domain` step installs the domain built
there (`eocene.domain.from`). The last section runs the laptop part of the domain pipeline.

**Run `01_quickstart.ipynb` first**: this notebook reads its configuration and output. Fields on the NEMO grid are
plotted in index space (`x`, `y`): the PALEORCA2 grid is not regular in longitude and latitude in the north.

In [ ]:
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

from ecefast.core.config import load_config
from ecefast.core.workspace import Context

# Working directory of the tutorials: 01_quickstart.ipynb writes the config and the outputs here
WORK = Path(os.environ.get("ECEFAST_TUTORIAL_DIR", Path(tempfile.gettempdir()) / "ecefast-tutorial"))
CONFIG = WORK / "laptop.yml"

# ctx expands path templates such as "{inputs.ece4_data}/oasis/{nemo.grid}/rstos.nc"
ctx = Context(load_config(str(CONFIG)))
import numpy as np

from ecefast.paleo.eocene.nemo.domain import EoceneDomain, check_vertical_grid
from ecefast.paleo.eocene.nemo.eoceneNEMO import EoceneNEMO
from ecefast.paleo.eocene.plots.helper_functions import plot_before_after


def open_both(template):
    """The file `template` (relative to the tree root) from the input and the output tree."""
    before = xr.open_dataset(ctx.expand("{inputs.ece4_data}/" + template))
    return before, xr.open_dataset(ctx.expand("{eocene.output_dir}/" + template))

## The `domain` step

In [ ]:
for kind in ("needs", "reads", "writes"):
    print(f"{kind}:")
    for template in getattr(EoceneDomain, kind):
        print("   ", ctx.expand(template) if kind != "needs" else template)

### Bathymetry

`bathy_metry` of the present-day domain (input tree) and of the Eocene domain (output tree); land is blank.

In [ ]:
present, eocene = open_both("nemo/domain/{nemo.grid}/domain_cfg.nc")
plot_before_after(
    present["bathy_metry"].where(present["bathy_metry"] > 0),
    eocene["bathy_metry"].where(eocene["bathy_metry"] > 0),
    "Bathymetry (m)",
    cmap="Blues",
);

In [ ]:
for name, ds in [("present day", present), ("Eocene", eocene)]:
    ocean = ds["bottom_level"] > 0
    print(
        f"{name:12s} ocean points: {int(ocean.sum())}, mean depth {float(ds['bathy_metry'].where(ocean).mean()):.0f} m"
    )

### Vertical grid check

`EoceneNEMO` builds the Eocene initial state on the present-day levels, so the two domains must have the same
vertical grid: the same number of levels and `nav_lev`, `e3t_1d` equal within `VERTICAL_GRID_ATOL`. The `domain`
step stops with an error otherwise; `check_vertical_grid` is the check it runs.

In [ ]:
check_vertical_grid(present, eocene, "present-day domain_cfg.nc", "Eocene domain_cfg.nc")
print("vertical grid: same")

fig, ax = plt.subplots(figsize=(5, 5))
for name, ds, style in [("present day", present, "o-"), ("Eocene", eocene, "x--")]:
    ax.plot(ds["e3t_1d"].squeeze(), ds["nav_lev"], style, label=name)
ax.invert_yaxis()
ax.set_xlabel("e3t_1d: layer thickness (m)")
ax.set_ylabel("nav_lev: depth (m)")
ax.legend();

The check also fails on a mismatch, for example a domain with one level less:

In [ ]:
try:
    check_vertical_grid(present, eocene.isel(z=slice(0, -1)), "present", "eocene (30 levels)")
except ValueError as error:
    print(error)

## The `nemo` step

In [ ]:
for kind in ("needs", "reads", "writes"):
    print(f"{kind}:")
    for template in getattr(EoceneNEMO, kind):
        print("   ", ctx.expand(template) if kind != "needs" else template)

### Tidal mixing (`zdfiwm_forcing_r720x360.nc`)

The internal-wave mixing forcing: one third of the Eocene M2 tidal dissipation (Green and Huber) goes into
`power_cri`, the other energy reservoirs are set to zero, and `bathymetry` is the Eocene ocean depth.

In [ ]:
before, after = open_both("nemo/initial/zdfiwm_forcing_r720x360.nc")
plot_before_after(
    np.log10(before["power_cri"].where(before["power_cri"] > 0)),
    np.log10(after["power_cri"].where(after["power_cri"] > 0)),
    "log10 power_cri",
    cmap="magma",
)
plot_before_after(before["bathymetry"], after["bathymetry"], "bathymetry", cmap="Blues_r");

### Geothermal heat flux (`Goutorbe_ghflux.nc`)

Computed from the Eocene sea-floor age (`age_depth_bath_55.xyadb` in the Herold folder) with an age-heat flow
relation, capped at 400 mW m⁻².

In [ ]:
before, after = open_both("nemo/initial/Goutorbe_ghflux.nc")
plot_before_after(before["gh_flux"], after["gh_flux"], "Geothermal heat flux", cmap="inferno");

### Ocean initial state

The present-day initial state is `woa13-levitus-L31.nc` (World Ocean Atlas). The step writes two Eocene variants
next to it, which the experiment chooses from: `_deepmip-30` (an idealised DeepMIP-like temperature profile) and
`_woa-pi` (the zonal mean of present-day WOA over the present-day ocean, the same at every longitude). Salinity is
34.7 everywhere in both. Surface temperature in January and the zonal mean
temperature section:

In [ ]:
woa = xr.open_dataset(ctx.expand("{inputs.ece4_data}/nemo/initial/woa13-levitus-L31.nc"))
variants = {
    name: xr.open_dataset(ctx.expand(f"{{eocene.output_dir}}/nemo/initial/woa13-levitus-L31_{name}.nc"))
    for name in ("deepmip-30", "woa-pi")
}
for name, ds in variants.items():
    plot_before_after(
        woa["thetao"].isel(time_counter=0, z=0),
        ds["thetao"].isel(time_counter=0, z=0),
        f"Surface temperature, January: woa13 vs {name}",
        labels=("Present day", f"Eocene {name}"),
        cmap="RdYlBu_r",
    )

In [ ]:
depth = {"z": present["nav_lev"].values}  # level depths (m), the same in both files
for name, ds in variants.items():
    plot_before_after(
        woa["thetao"].isel(time_counter=0).mean("x").assign_coords(depth),
        ds["thetao"].isel(time_counter=0).mean("x").assign_coords(depth),
        f"Zonal mean temperature, January: woa13 vs {name} (z: depth in m, y: grid row)",
        labels=("Present day", f"Eocene {name}"),
        cmap="RdYlBu_r",
        yincrease=False,
    )

### River runoff (`runoff_DaiTrenberth_PALEORCA2.nc`)

The step sets the runoff (`sorunoff`) and the runoff coefficient (`socoefr`) to zero everywhere. Annual mean
runoff:

In [ ]:
before, after = open_both("nemo/climatology/runoff_DaiTrenberth_PALEORCA2.nc")
plot_before_after(
    before["sorunoff"].mean("time"), after["sorunoff"].mean("time"), "Annual mean runoff", cmap="Blues", vmax=1e-4
)
print("Eocene: sorunoff max", float(after["sorunoff"].max()), "| socoefr max", float(after["socoefr"].max()))

## The NEMO domain pipeline (`ecefast nemo domain`)

The Eocene domain installed above was built on Atos by the NEMO domain pipeline, configured in the `nemo.domain`
section of the configuration file. On a laptop the pipeline runs its first steps: the PALEORCA2 coordinates, the
bathymetry remapped to the PALEORCA2 grid, and the DOMAINcfg namelist; DOMAINcfg and the steps after it are off
(`nemo.domain.steps`). `--bathymetry eocene` builds the Eocene bathymetry only (a few seconds). The command writes
only to `nemo.domain.output_dir`: final files under `nemo/domain/<grid>/<name>/`, as in an EC-Earth input tree,
and intermediate files (coordinates, bathymetries, DOMAINcfg namelists) under `work/`.

In [ ]:
!ecefast nemo domain -c {CONFIG} --bathymetry eocene

In [ ]:
domain_out = Path(ctx.config.nemo.domain.output_dir)
for path in sorted(domain_out.rglob("*")):
    if path.is_file():
        print(path.relative_to(domain_out))

The remapped Herold bathymetry (input of DOMAINcfg) next to the bathymetry of the domain DOMAINcfg built from it
on Atos:

In [ ]:
remapped = xr.open_dataset(next(domain_out.glob("work/bathymetry/*/HEROLD_*.nc")))["bathy_metry"]
plot_before_after(
    remapped.where(remapped > 0),
    eocene["bathy_metry"].where(eocene["bathy_metry"] > 0),
    "Eocene bathymetry (m)",
    labels=("Remapped Herold bathymetry", "Eocene domain_cfg.nc (DOMAINcfg)"),
    cmap="Blues",
);